In [ ]:
from pathlib import Path
import torch
import re
import datetime

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
database_path = Path("database")
documents = []

for md_file in database_path.glob("*.md"):
    loader = TextLoader(str(md_file), encoding="utf-8")
    documents.extend(loader.load())

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

vectordb = FAISS.from_documents(chunks, embedding)

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt, max_length=4000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

In [ ]:
date = datetime.date.today().strftime("%d %m %Y")

In [ ]:
dream_title = ResponseSchema(
    name="dream-title",
    description="A concise meaningful title the discribe the dream. Mainly consists of three or less words."
)
dream_date = ResponseSchema(
    name="dream_date",
    description="The date (in DD-MM-YYYY format) in which the dream happened. {date}"
)
dream_desc = ResponseSchema(
    name="dream_description",
    description="A exact replica of the user inputed dream word for word."
)
dream_symbols = ResponseSchema(
    name="dream_symbols",
    description="A list of all the possible symbolism cotained in the dream."
)
dream_vibes = ResponseSchema(
    name="dream_vibes",
    description="A short list of words that discribe the main feeling (vibe) of the dream."
)

response_schemas = [dream_title,
                    dream_date,
                    dream_desc,
                    dream_symbols, 
                    dream_vibes]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [ ]:
dream_journal_template_prompt = """
You are an expert dream journaler and analyst that extracts dream details based on the user's input.



Extract all qualifications as follows:

dream title
dream date {date}
dream description, including details about what happened, how it felt, and any related ideas
dream symbols
dream vibes, describing the main feeling of the dream


Respond ONLY in Markdown format as follows:
{format_instructions}

Example Input:
"
I had a dream were I was runing away from a mirror
"

Expected output (im markdown):
"
# Dream Title
The Mirror

**Dream Date:** 27-07-2026

## Dream

## Dream Description
I had a dream were I was runing away from a mirror

## Dream Symbols
- Mirror
- Avoidance

## Dream Vibes
- Fear
- Avoidance
"

Now extract from the following input:
"{user_input}"
"""

In [ ]:
def ask_question(query):
    docs = vectordb.similarity_search(query, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    prompt = f"""You are a helpful assistant. Use the following context to answer the question. Question: {query}"""
    
    result = generate_text(prompt)
    return result.strip()

In [ ]:
user_input = "I had a dream where I was eating then suddenly I turned into a 100m tall gaint!"

prompt = PromptTemplate(
    template=dream_journal_template_prompt,
    input_variables=["user_input", "format_instructions"]
).format(user_input=user_input, format_instructions=format_instructions)

In [ ]:
answer = ask_question(prompt)
print(answer)